In [2]:
from cellgrn.main import normalize_rna,parse_edges,compute_all_cells_grn,summarize_grn,format_sample_grn,format_celltype_grn
from cellgrn.utils import grn_umap
import numpy as np
import pandas as pd
import os
import anndata as ad
import pickle
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import sparse

/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [ ]:


input_gene = [i.rstrip() for i in open("/home/shaliu_fu/multireg/multigrn/input_data/all_gene/10X_PBMC/input_gene.txt")]
input_peak = [i.rstrip() for i in open("/home/shaliu_fu/multireg/multigrn/input_data/all_gene/10X_PBMC/input_peak.txt")]
input_tf =  [i.rstrip() for i in open("/home/shaliu_fu/multireg/multigrn/input_data/all_gene/10X_PBMC/input_tf.txt")]

cell_meta = pd.read_csv("/home/shaliu_fu/multireg/multigrn/input_data/bench_dataset/10X_PBMC//metadata_pstime.csv")
cell_meta.index = cell_meta['barcode']
input_rna = ad.read_h5ad(f"/home/shaliu_fu/multireg/benchmark/bench_dataset/10X_PBMC/PBMC-multiome-raw-RNA-counts.h5ad")
input_atac = ad.read_h5ad(f"/home/shaliu_fu/multireg/benchmark/bench_dataset/10X_PBMC/PBMC-multiome-raw-ATAC-peaks.h5ad")

cell_types = cell_meta['annotated_labels']


In [ ]:

for soft in ["linger","scenic2"]:
# for soft in ["scenic2"]:

    input_df1 = pd.DataFrame(input_rna.X.toarray(),index=input_rna.obs.index.values,columns=input_rna.var.index.values)
    peak_rename = [i.replace(":","-") for i in input_atac.var.index.values]
    input_df2 = pd.DataFrame(input_atac.X.toarray(),index=input_atac.obs.index.values,columns=peak_rename)

    outdir = f"../output/res_pbmc_{soft}/"
    os.system(f"mkdir -p {outdir}")


    cand_df = pd.read_csv(f"/home/shaliu_fu/multireg/cellGRN/data/10X_PBMC/{soft}_grn.csv",header=0)

    input_genes = [i.rstrip() for i in open(f"/home/shaliu_fu/multireg/cellGRN/data/10X_PBMC/{soft}_genes.txt")]
    input_peaks = [i.rstrip() for i in open(f"/home/shaliu_fu/multireg/cellGRN/data/10X_PBMC/{soft}_peaks.txt")]

    input_peaks = [i.replace(":","-") for i in input_peaks]
    input_df1 = input_df1[input_genes]
    input_df2 = input_df2[input_peaks]


    rna_data1,rna_data2 = normalize_rna(input_df1)
    atac_data = input_df2.copy()

    input_tfs = [tf for tf in input_tf if tf in input_genes]
    tf_data1 = rna_data1[input_tfs].copy()
    tf_data2 = rna_data2[input_tfs].copy()

    edges_idx,edges_name = parse_edges(cand_df, input_tfs, input_genes, input_peaks)

    grn_scale2 = compute_all_cells_grn(tf_data2, rna_data2, atac_data,edges_idx, edges_name,
        input_tfs, input_genes, input_peaks)

    with open(f"{outdir}/{soft}_cell_grn.pkl", "wb") as f:
        pickle.dump(grn_scale2, f)

    sample_grn_scale2, celltype_grn_scale2 = summarize_grn(grn_scale2, cell_types)


    tf_gene_res_scale2, tf_peak_res_scale2, gene_peak_res_scale2 = format_sample_grn(sample_grn_scale2)
    tf_gene_ct_res_scale2, tf_peak_ct_res_scale2, gene_peak_ct_res_scale2 = format_celltype_grn(celltype_grn_scale2)

    

    tf_gene_res_scale2.to_csv(os.path.join(outdir, "tf_gene_sample_scale2.csv"), index=False)
    tf_peak_res_scale2.to_csv(os.path.join(outdir, "tf_peak_sample_scale2.csv"), index=False)
    gene_peak_res_scale2.to_csv(os.path.join(outdir, "gene_peak_sample_scale2.csv"), index=False)

    tf_gene_ct_res_scale2.to_csv(os.path.join(outdir, "tf_gene_celltype_scale2.csv"), index=False)
    tf_peak_ct_res_scale2.to_csv(os.path.join(outdir, "tf_peak_celltype_scale2.csv"), index=False)
    gene_peak_ct_res_scale2.to_csv(os.path.join(outdir, "gene_peak_celltype_scale2.csv"), index=False)